# DataStory & Drift AI Engine — POC de Fast Prompting

**Estudiante:** Lucio Benitez  
**Curso:** Generación de Prompts  
**Comisión:** `96090`  
**Entrega:** Preentrega 2

## Objetivo de la notebook
Demostrar cómo una combinación de validaciones en Python y Fast Prompting puede traducir señales de Model Drift y XAI a lenguaje ejecutivo, minimizando alucinaciones, exposición de datos sensibles y consultas innecesarias a una API.

## 1. Diseño de la POC

Flujo propuesto:

**Métricas agregadas → validación Python → cálculo determinístico → prompt → LLM → validación de salida → revisión humana**

Principios:
- Python resuelve cálculos exactos.
- El LLM no puede inventar métricas ni valores financieros.
- SHAP se trata como explicación del comportamiento del modelo, no como causalidad.
- Los datos son sintéticos y agregados.
- El experimento A/B usa 2 llamadas; el flujo final usa 1 llamada por caso.

In [1]:
# Si es necesario, descomentar en un entorno nuevo:
# %pip install -q openai pandas

import os
import re
import json
from getpass import getpass

import pandas as pd

## 2. Caso sintético de prueba

Los datos de negocio son **sintéticos** y existen únicamente para demostrar la POC.
No deben interpretarse como información de una empresa real.

In [2]:
case = {
    "model": "Predicción de churn de clientes bancarios",
    "period": "últimos 2 meses",
    "psi_before": 0.08,
    "psi_after": 0.28,
    "accuracy_drop_pct": 14.0,
    "changed_shap_factors": [
        "Frecuencia de transacciones internacionales",
        "Ratio de inflación percibida"
    ],
    "business_effect": "Aumento de falsos negativos en clientes VIP",

    # Variables sintéticas para estimar exposición económica.
    "vip_clients_at_risk": 120,
    "avg_monthly_margin_usd": 350,
    "retraining_cost_usd": 1800,

    "data_origin": "sintético / demostración académica"
}

case

{'model': 'Predicción de churn de clientes bancarios',
 'period': 'últimos 2 meses',
 'psi_before': 0.08,
 'psi_after': 0.28,
 'accuracy_drop_pct': 14.0,
 'changed_shap_factors': ['Frecuencia de transacciones internacionales',
  'Ratio de inflación percibida'],
 'business_effect': 'Aumento de falsos negativos en clientes VIP',
 'vip_clients_at_risk': 120,
 'avg_monthly_margin_usd': 350,
 'retraining_cost_usd': 1800,
 'data_origin': 'sintético / demostración académica'}

## 3. Controles de entrada

Se bloquean identificadores personales y se validan rangos antes de enviar contenido al LLM.

In [3]:
SENSITIVE_KEYS = {
    "name", "nombre", "email", "dni", "documento",
    "account_number", "numero_cuenta", "client_id", "customer_id",
    "phone", "telefono", "address", "direccion"
}

def validate_no_sensitive_fields(data):
    suspicious = [k for k in data.keys() if k.lower() in SENSITIVE_KEYS]
    if suspicious:
        raise ValueError(
            f"Se detectaron campos potencialmente sensibles: {suspicious}. "
            "La POC solo admite datos agregados o sintéticos."
        )

def validate_metrics(data):
    required = [
        "psi_before", "psi_after", "accuracy_drop_pct",
        "changed_shap_factors", "business_effect"
    ]
    missing = [k for k in required if k not in data]
    if missing:
        raise ValueError(f"Faltan campos obligatorios: {missing}")

    if data["psi_before"] < 0 or data["psi_after"] < 0:
        raise ValueError("PSI no puede ser negativo.")

    if not 0 <= data["accuracy_drop_pct"] <= 100:
        raise ValueError("accuracy_drop_pct debe estar entre 0 y 100.")

    if not isinstance(data["changed_shap_factors"], list):
        raise TypeError("changed_shap_factors debe ser una lista.")

validate_no_sensitive_fields(case)
validate_metrics(case)
print("Entrada validada correctamente.")

Entrada validada correctamente.


## 4. Cálculos determinísticos

Para evitar alucinaciones aritméticas, los valores financieros se calculan en Python y se entregan ya resueltos al modelo.

**Importante:** esta estimación representa una exposición teórica basada en las variables sintéticas cargadas. No supone que todos los clientes en riesgo vayan a abandonar.

In [4]:
business_metrics = {
    "maximum_monthly_margin_exposure_usd": (
        case["vip_clients_at_risk"] * case["avg_monthly_margin_usd"]
    ),
    "retraining_cost_usd": case["retraining_cost_usd"]
}

business_metrics

{'maximum_monthly_margin_exposure_usd': 42000, 'retraining_cost_usd': 1800}

## 5. Clasificación orientativa de PSI

Para la POC se define una política interna simplificada:
- `< 0.10`: cambio bajo;
- `0.10–0.25`: cambio moderado;
- `> 0.25`: cambio alto.

Estos umbrales se usan únicamente como regla de demostración y deben ser ajustados por el equipo responsable según el dominio y la política de monitoreo real.

In [5]:
def psi_status(value):
    if value < 0.10:
        return "bajo"
    if value <= 0.25:
        return "moderado"
    return "alto"

psi_analysis = {
    "before": psi_status(case["psi_before"]),
    "after": psi_status(case["psi_after"]),
    "delta": round(case["psi_after"] - case["psi_before"], 3)
}

psi_analysis

{'before': 'bajo', 'after': 'alto', 'delta': 0.2}

## 6. Prompt A — Baseline

Este prompt conserva la lógica general de la primera entrega. Sirve como punto de comparación.

In [6]:
def build_baseline_prompt(case, business_metrics):
    return f"""
Actúa como un Lead Data Scientist y Strategy Consultant.
Explica al comité ejecutivo el siguiente deterioro de modelo:

Modelo: {case['model']}
PSI: {case['psi_before']} -> {case['psi_after']}
Caída de precisión: {case['accuracy_drop_pct']}%
Factores SHAP que cambiaron: {", ".join(case['changed_shap_factors'])}
Impacto observado: {case['business_effect']}
Exposición máxima mensual calculada: USD {business_metrics['maximum_monthly_margin_exposure_usd']}
Costo de reentrenamiento: USD {business_metrics['retraining_cost_usd']}

Entrega:
1. Resumen ejecutivo.
2. Analogía para explicar el drift.
3. Impacto de negocio.
4. Plan de acción y ROI.
""".strip()

prompt_a = build_baseline_prompt(case, business_metrics)
print(prompt_a)

Actúa como un Lead Data Scientist y Strategy Consultant.
Explica al comité ejecutivo el siguiente deterioro de modelo:

Modelo: Predicción de churn de clientes bancarios
PSI: 0.08 -> 0.28
Caída de precisión: 14.0%
Factores SHAP que cambiaron: Frecuencia de transacciones internacionales, Ratio de inflación percibida
Impacto observado: Aumento de falsos negativos en clientes VIP
Exposición máxima mensual calculada: USD 42000
Costo de reentrenamiento: USD 1800

Entrega:
1. Resumen ejecutivo.
2. Analogía para explicar el drift.
3. Impacto de negocio.
4. Plan de acción y ROI.


## 7. Prompt B — Fast Prompting optimizado

Mejoras incorporadas:
- rol claro;
- contexto delimitado;
- grounding;
- reglas contra causalidad indebida;
- few-shot;
- descomposición de tareas;
- formato JSON;
- auto-chequeo dentro de la misma consulta;
- prompts visuales en la misma salida.

In [7]:
FEW_SHOT = """
EJEMPLO DE TRADUCCIÓN:
Señal técnica: "Una variable aumentó su importancia SHAP".
Traducción correcta: "El modelo está apoyándose más en esa señal para producir sus predicciones."
Traducción incorrecta: "Esa variable causó el comportamiento del cliente."
"""

def build_optimized_prompt(case, business_metrics, psi_analysis):
    payload = {
        "caso": case,
        "metricas_negocio_calculadas_por_python": business_metrics,
        "politica_psi_poc": psi_analysis
    }

    return f"""
[ROL]
Actúa como Lead Data Scientist & Strategy Consultant especializado en
AI Governance y Executive Data Storytelling.

[OBJETIVO]
Transformar métricas técnicas ya validadas en un memo ejecutivo breve
y en dos prompts visuales conceptuales.

[DATOS AUTORIZADOS]
--- INICIO DATOS ---
{json.dumps(payload, ensure_ascii=False, indent=2)}
--- FIN DATOS ---

[REGLAS DE GROUNDING]
1. Usa únicamente la información dentro de DATOS AUTORIZADOS.
2. No inventes métricas, porcentajes, causas, fechas, clientes ni importes.
3. Si un dato no está disponible, escribe exactamente: "NO DISPONIBLE".
4. No presentes SHAP como causalidad. SHAP describe contribución/importancia
   dentro del modelo, no una relación causal con el comportamiento real.
5. No conviertas una caída de precisión en una pérdida monetaria directa.
6. La "maximum_monthly_margin_exposure_usd" es exposición máxima teórica,
   no pérdida confirmada.
7. No incluyas datos personales ni solicites identificadores.
8. Toda recomendación requiere validación humana antes de una decisión real.

{FEW_SHOT}

[TAREAS]
A. Resume qué cambió y por qué merece atención ejecutiva.
B. Crea una analogía del mundo real para explicar Model Drift sin usar fórmulas.
C. Explica el impacto observado y separa claramente:
   - hechos medidos;
   - estimaciones;
   - limitaciones.
D. Propón máximo 3 acciones.
E. Genera un prompt visual conceptual para Model Drift.
F. Genera un prompt visual conceptual para XAI.

[RESTRICCIONES VISUALES]
- No pedir números, ejes, dashboards ni gráficos cuantitativos inventados.
- Evitar texto pequeño o ilegible.
- Model Drift: mostrar "distribución estable -> desplazamiento".
- XAI: mostrar "señal del modelo -> capa de explicación -> decisión humana".
- Estética profesional, ejecutiva, limpia, 16:9.

[AUTO-CHEQUEO INTERNO]
Antes de responder verifica:
- ¿Todos los números provienen de los datos?
- ¿Se evitó causalidad indebida?
- ¿Se distinguieron hechos y estimaciones?
- ¿Se indicó revisión humana?
- ¿Los prompts visuales son conceptuales y no aparentan evidencia cuantitativa?
No muestres este razonamiento; devuelve solamente el resultado final.

[FORMATO DE SALIDA]
Devuelve SOLO JSON válido con esta estructura:
{{
  "status_validacion": "APTO o REVISAR",
  "resumen_ejecutivo": "...",
  "analogia_negocio": "...",
  "impacto": {{
    "hechos_medidos": ["..."],
    "estimaciones": ["..."],
    "limitaciones": ["..."]
  }},
  "plan_accion": ["...", "...", "..."],
  "prompt_imagen_model_drift": "...",
  "prompt_imagen_xai": "...",
  "requiere_revision_humana": true
}}
""".strip()

prompt_b = build_optimized_prompt(case, business_metrics, psi_analysis)
print(prompt_b)

[ROL]
Actúa como Lead Data Scientist & Strategy Consultant especializado en
AI Governance y Executive Data Storytelling.

[OBJETIVO]
Transformar métricas técnicas ya validadas en un memo ejecutivo breve
y en dos prompts visuales conceptuales.

[DATOS AUTORIZADOS]
--- INICIO DATOS ---
{
  "caso": {
    "model": "Predicción de churn de clientes bancarios",
    "period": "últimos 2 meses",
    "psi_before": 0.08,
    "psi_after": 0.28,
    "accuracy_drop_pct": 14.0,
    "changed_shap_factors": [
      "Frecuencia de transacciones internacionales",
      "Ratio de inflación percibida"
    ],
    "business_effect": "Aumento de falsos negativos en clientes VIP",
    "vip_clients_at_risk": 120,
    "avg_monthly_margin_usd": 350,
    "retraining_cost_usd": 1800,
    "data_origin": "sintético / demostración académica"
  },
  "metricas_negocio_calculadas_por_python": {
    "maximum_monthly_margin_exposure_usd": 42000,
    "retraining_cost_usd": 1800
  },
  "politica_psi_poc": {
    "before": "ba

## 8. Configuración opcional de la API

La clave **no se escribe en la notebook**. Se solicita de forma interactiva.

Para revisar la estructura sin gastar API, mantener `RUN_API = False`.
Para ejecutar la demostración real, cambiarlo a `True`.

> La Responses API permite recuperar el uso de tokens de la respuesta, lo que facilita registrar el costo técnico de cada experimento.

In [8]:
RUN_API = False
MODEL = "gpt-5.6-luna"

api_calls = 0
usage_log = []

if RUN_API:
    from openai import OpenAI

    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

    client = OpenAI()
else:
    client = None

print("Modo:", "API REAL" if RUN_API else "DRY RUN")

Modo: DRY RUN


In [9]:
def call_llm(prompt, label):
    global api_calls

    if not RUN_API:
        print(f"[DRY RUN] {label}: no se realizó ninguna consulta.")
        return None

    response = client.responses.create(
        model=MODEL,
        input=prompt,
        text={"verbosity": "low"}
    )

    api_calls += 1

    usage = {
        "label": label,
        "input_tokens": getattr(response.usage, "input_tokens", None),
        "output_tokens": getattr(response.usage, "output_tokens", None),
        "total_tokens": getattr(response.usage, "total_tokens", None),
    }
    usage_log.append(usage)

    return response.output_text

## 9. Experimento A/B

Durante el experimento se realizan como máximo **2 consultas**:
1. Prompt A — baseline.
2. Prompt B — optimizado.

En operación real se conserva únicamente Prompt B, por lo que el costo baja a **1 consulta por caso**.

In [10]:
output_a = call_llm(prompt_a, "baseline")
output_b = call_llm(prompt_b, "optimized")

print("Consultas realizadas:", api_calls)

[DRY RUN] baseline: no se realizó ninguna consulta.
[DRY RUN] optimized: no se realizó ninguna consulta.
Consultas realizadas: 0


## 10. Parseo y control de la salida optimizada

In [11]:
def parse_json_output(text):
    if text is None:
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Fallback básico si el modelo agrega fences markdown.
        cleaned = re.sub(r"^```json\s*|\s*```$", "", text.strip(), flags=re.IGNORECASE)
        return json.loads(cleaned)

def validate_optimized_output(result):
    if result is None:
        return {"status": "NO EJECUTADO", "issues": []}

    issues = []
    required = {
        "status_validacion",
        "resumen_ejecutivo",
        "analogia_negocio",
        "impacto",
        "plan_accion",
        "prompt_imagen_model_drift",
        "prompt_imagen_xai",
        "requiere_revision_humana"
    }

    missing = sorted(required - set(result.keys()))
    if missing:
        issues.append(f"Faltan campos: {missing}")

    if result.get("requiere_revision_humana") is not True:
        issues.append("Debe requerir revisión humana.")

    text = json.dumps(result, ensure_ascii=False).lower()

    # Control simple de causalidad indebida alrededor de SHAP.
    risky_phrases = [
        "shap demuestra que",
        "shap prueba que",
        "shap confirma que",
        "causado por shap"
    ]
    for phrase in risky_phrases:
        if phrase in text:
            issues.append(f"Posible causalidad indebida: '{phrase}'")

    return {
        "status": "APTO" if not issues else "REVISAR",
        "issues": issues
    }

parsed_b = parse_json_output(output_b)
validation_b = validate_optimized_output(parsed_b)
validation_b

{'status': 'NO EJECUTADO', 'issues': []}

## 11. Comparación de configuraciones

No se utiliza otro LLM como juez porque agregaría una tercera consulta.
La evaluación combina criterios observables y revisión humana.

In [12]:
comparison = pd.DataFrame([
    {
        "criterio": "Rol y contexto delimitado",
        "Prompt A": "Parcial",
        "Prompt B": "Sí"
    },
    {
        "criterio": "Grounding explícito",
        "Prompt A": "No",
        "Prompt B": "Sí"
    },
    {
        "criterio": "Control de causalidad SHAP",
        "Prompt A": "No",
        "Prompt B": "Sí"
    },
    {
        "criterio": "Valores financieros calculados en Python",
        "Prompt A": "Sí",
        "Prompt B": "Sí"
    },
    {
        "criterio": "Formato estructurado",
        "Prompt A": "No",
        "Prompt B": "JSON"
    },
    {
        "criterio": "Self-check sin llamada extra",
        "Prompt A": "No",
        "Prompt B": "Sí"
    },
    {
        "criterio": "Prompts visuales incluidos",
        "Prompt A": "No",
        "Prompt B": "Sí"
    },
])

comparison

,criterio,Prompt A,Prompt B
0,Rol y contexto delimitado,Parcial,Sí
1,Grounding explícito,No,Sí
2,Control de causalidad SHAP,No,Sí
3,Valores financieros calculados en Python,Sí,Sí
4,Formato estructurado,No,JSON
5,Self-check sin llamada extra,No,Sí
6,Prompts visuales incluidos,No,Sí


### Rúbrica manual de eficacia

Después de ejecutar ambos prompts, puntuar de 1 a 5:
- claridad ejecutiva;
- fidelidad a los datos;
- ausencia de causalidad indebida;
- accionabilidad;
- concisión;
- cumplimiento del formato.

Registrar los resultados reales en la tabla siguiente.

In [13]:
rubric = pd.DataFrame({
    "criterio": [
        "Claridad ejecutiva",
        "Fidelidad a los datos",
        "Ausencia de causalidad indebida",
        "Accionabilidad",
        "Concisión",
        "Cumplimiento de formato"
    ],
    "Prompt A (1-5)": [None] * 6,
    "Prompt B (1-5)": [None] * 6
})

rubric

,criterio,Prompt A (1-5),Prompt B (1-5)
0,Claridad ejecutiva,None,None
1,Fidelidad a los datos,None,None
2,Ausencia de causalidad indebida,None,None
3,Accionabilidad,None,None
4,Concisión,None,None
5,Cumplimiento de formato,None,None


## 12. Registro de uso y costos

La POC registra tokens y cantidad de consultas. Para un cálculo monetario exacto, multiplicar el uso por la tarifa vigente del modelo elegido al momento de ejecutar la notebook.

Esto evita dejar un precio fijo que pueda quedar desactualizado.

In [14]:
usage_df = pd.DataFrame(usage_log)
usage_df

""


In [15]:
print(f"Total de consultas API ejecutadas: {api_calls}")
print("Objetivo en producción: 1 consulta por caso.")

Total de consultas API ejecutadas: 0
Objetivo en producción: 1 consulta por caso.


## 13. Iteración de prompts visuales

### Model Drift
**V1:** estética futurista + dashboards + gráficos.  
**Riesgo:** números/ejes inventados y texto ilegible.  
**V2:** imagen conceptual 16:9, sin cifras ni gráficos cuantitativos, enfocada en el desplazamiento de una distribución.

### XAI
**V1:** esfera con red neuronal.  
**Riesgo:** imagen atractiva pero genérica.  
**V2:** tres etapas visuales: señal del modelo → explicación → decisión humana; sin causalidad ni texto pequeño.

Si `Prompt B` fue ejecutado, los prompts finales aparecen en:
- `parsed_b["prompt_imagen_model_drift"]`
- `parsed_b["prompt_imagen_xai"]`

In [16]:
if parsed_b:
    print("MODEL DRIFT:\n", parsed_b.get("prompt_imagen_model_drift"))
    print("\nXAI:\n", parsed_b.get("prompt_imagen_xai"))
else:
    print("Ejecutar Prompt B para obtener los prompts visuales finales.")

Ejecutar Prompt B para obtener los prompts visuales finales.


## 14. Criterios de aprobación

La POC se considera aprobada cuando:
1. la entrada no contiene datos sensibles;
2. las métricas pasan las validaciones;
3. el Prompt B devuelve la estructura requerida;
4. no inventa cifras;
5. no trata SHAP como causalidad;
6. diferencia hechos, estimaciones y limitaciones;
7. exige revisión humana;
8. el flujo final utiliza una sola consulta por caso.

---

## 15. Conclusión

Fast Prompting mejora la propuesta inicial porque no se limita a agregar instrucciones: organiza el contexto, restringe el espacio de respuesta, incluye ejemplos, define un formato verificable y combina el LLM con controles determinísticos.

La arquitectura resultante es más clara, reproducible, económica y segura para una POC de comunicación ejecutiva de Model Drift y XAI.